<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/W07_overfitting_augmentation_sota.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision: Overfitting, Data Augmentation und SOTA-Modelle

In diesem Notebook erlebst du drei zentrale Konzepte moderner Computer Vision:

1. **Overfitting** – Was passiert, wenn ein Modell die Trainingsdaten „auswendig lernt"?  
2. **Data Augmentation** – Wie holt man mehr aus wenigen Bildern heraus?  
3. **SOTA-Modelle** – Wie nutzt man vortrainierte Spitzenmodelle von Hugging Face und YOLO in wenigen Zeilen Code?

> 💡 **Tipp:** Aktiviere in Colab unter *Laufzeit → Laufzeittyp ändern* die GPU, damit das CNN-Training schneller läuft.

In [ ]:
# Pakete installieren (nur beim ersten Start nötig)
!pip install -q ultralytics transformers timm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print("PyTorch Version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Gerät: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Teil 1: Overfitting – Wenn das Modell zu viel auswendig lernt

### Was ist Overfitting?

Stell dir vor, du bereitest dich auf eine Prüfung vor, indem du ausschliesslich die Musterprüfungen der letzten Jahre Satz für Satz auswendig lernst – ohne die Konzepte zu verstehen.  
In der echten Prüfung (neue Fragen, ähnliche Konzepte) schneidest du dann schlecht ab.

Genau das passiert beim maschinellen Lernen: Das Modell „memoriert" die Trainingsdaten,  
kann aber keine allgemeingültigen Muster lernen.

**Erkennungszeichen in den Losskurven:**
| Kurve | Overfitting-Muster |
|---|---|
| Training Loss (blau) | sinkt kontinuierlich ✓ |
| Validierungs-Loss (rot) | sinkt zunächst, steigt dann wieder ✗ |

Wir demonstrieren das mit **CIFAR-10** (60 000 Farbbilder, 32×32 Pixel, 10 Klassen)  
– und verwenden bewusst nur **500 Trainingsbilder**, damit Overfitting klar sichtbar wird.

In [ ]:
# CIFAR-10 laden
KLASSEN_DE = ['Flugzeug','Auto','Vogel','Katze','Reh','Hund','Frosch','Pferd','Schiff','LKW']
KLASSEN_EN = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

N_TRAIN = 500   # Bewusst klein → Overfitting!
N_VAL   = 1000

transform_basis = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=transform_basis)
full_val   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_basis)

# Rohes Dataset (PIL-Bilder) für Visualisierungen
full_train_raw = torchvision.datasets.CIFAR10('./data', train=True,  download=False)
full_val_raw   = torchvision.datasets.CIFAR10('./data', train=False, download=False)

train_loader = DataLoader(Subset(full_train, range(N_TRAIN)), batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(Subset(full_val,   range(N_VAL)),   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Trainingsbilder:     {N_TRAIN}")
print(f"Validierungsbilder:  {N_VAL}")

# Je ein Beispielbild pro Klasse
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
gezeigt = {}
i = 0
idx = 0
while i < 10:
    img_pil, label = full_train_raw[idx]
    idx += 1
    if label not in gezeigt:
        gezeigt[label] = True
        axes[i // 5][i % 5].imshow(img_pil)
        axes[i // 5][i % 5].set_title(KLASSEN_DE[label], fontsize=11)
        axes[i // 5][i % 5].axis('off')
        i += 1
plt.suptitle("CIFAR-10 – ein Beispielbild pro Klasse", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Kleines CNN – ohne Regularisierung (Dropout etc.)
class KleinesCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   #  8x8
            nn.Conv2d(64,128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   #  4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

params = sum(p.numel() for p in KleinesCNN().parameters())
print(f"Modellparameter: {params:,}  (~{params/1e6:.2f}M)")

In [ ]:
def trainiere(modell, train_ldr, val_ldr, n_epochs=60, lr=1e-3):
    """Trainiert das Modell und gibt den Verlauf als Dict zurück."""
    opt    = optim.Adam(modell.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    verlauf = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[])

    for ep in range(1, n_epochs + 1):
        # --- Training ---
        modell.train()
        t_loss = t_ok = t_n = 0
        for x, y in train_ldr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out  = modell(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()
            t_loss += loss.item() * len(y)
            t_ok   += (out.argmax(1) == y).sum().item()
            t_n    += len(y)

        # --- Validierung ---
        modell.eval()
        v_loss = v_ok = v_n = 0
        with torch.no_grad():
            for x, y in val_ldr:
                x, y = x.to(device), y.to(device)
                out   = modell(x)
                v_loss += loss_fn(out, y).item() * len(y)
                v_ok   += (out.argmax(1) == y).sum().item()
                v_n    += len(y)

        verlauf['train_loss'].append(t_loss / t_n)
        verlauf['val_loss'  ].append(v_loss / v_n)
        verlauf['train_acc' ].append(t_ok   / t_n)
        verlauf['val_acc'   ].append(v_ok   / v_n)

        if ep % 15 == 0 or ep == 1:
            print(f"Ep {ep:3d}/{n_epochs} | "
                  f"Train  Loss={verlauf['train_loss'][-1]:.3f}  Acc={verlauf['train_acc'][-1]:.1%} | "
                  f"Val    Loss={verlauf['val_loss'][-1]:.3f}  Acc={verlauf['val_acc'][-1]:.1%}")
    return verlauf

In [ ]:
N_EPOCHS = 60
print("=== Training OHNE Augmentation  (500 Trainingsbilder) ===")
modell_1 = KleinesCNN().to(device)
v1 = trainiere(modell_1, train_loader, val_loader, n_epochs=N_EPOCHS)

In [ ]:
# Losskurven plotten und Overfitting-Punkt markieren
ep = range(1, N_EPOCHS + 1)
best = int(np.argmin(v1['val_loss']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(ep, v1['train_loss'], color='steelblue', lw=2, label='Training Loss')
ax1.plot(ep, v1['val_loss'],   color='tomato',    lw=2, label='Validierungs-Loss')
ax1.axvline(best + 1, color='orange', ls='--', lw=1.5, label=f'Bestes Val (Ep. {best+1})')
ax1.annotate('Overfitting\nbeginnt',
             xy=(best + 1, v1['val_loss'][best]),
             xytext=(best + 7, v1['val_loss'][best] + 0.04),
             arrowprops=dict(arrowstyle='->', color='darkorange'),
             fontsize=9, color='darkorange')
ax1.set_xlabel('Epoche'); ax1.set_ylabel('Loss')
ax1.set_title('Loss-Kurven'); ax1.legend(); ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(ep, [a*100 for a in v1['train_acc']], color='steelblue', lw=2, label='Training Acc.')
ax2.plot(ep, [a*100 for a in v1['val_acc']],   color='tomato',    lw=2, label='Validierungs-Acc.')
ax2.set_xlabel('Epoche'); ax2.set_ylabel('Accuracy (%)'); ax2.set_ylim(0, 105)
ax2.set_title('Accuracy-Kurven'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Overfitting: Training auf 500 Bildern ohne Augmentation", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📊 Finale Werte:")
print(f"  Training Acc.:      {v1['train_acc'][-1]:.1%}")
print(f"  Validierungs-Acc.:  {v1['val_acc'][-1]:.1%}")
print(f"  → Grosse Lücke zwischen Training und Validierung = Overfitting!")

### 🔍 Interpretation der Kurven

- **Training Loss** (blau) sinkt stetig gegen 0 → das Modell hat die 500 Bilder fast auswendig gelernt
- **Validierungs-Loss** (rot) sinkt zunächst, steigt dann aber wieder an → klassisches Overfitting-Muster

Die **orangene Linie** markiert den optimalen Trainingszeitpunkt.  
Alles Training danach macht das Modell auf neuen Daten schlechter – nicht besser.

> **Early Stopping** ist eine einfache Gegenmassnahme: Training abbrechen, sobald der Validierungs-Loss  
> nicht mehr sinkt. In Teil 2 sehen wir eine weitere, sehr wirkungsvolle Methode.

---
## Teil 2: Data Augmentation – Mehr aus wenigen Daten herausholen

### Die Grundidee

Wenn ein Modell nur 500 Bilder sieht, kennt es z.B. einen Hund nur in einer bestimmten  
Belichtung, von einem bestimmten Winkel, in einem bestimmten Ausschnitt.

**Data Augmentation** erzeugt bei jedem Trainingsschritt **zufällige Varianten** der Originalbilder:

| Transformation | Wirkung |
|---|---|
| Horizontales Spiegeln | Ein Hund von links = ein Hund von rechts |
| Zufälliger Ausschnitt | Verschiedene Ausschnitte → Positionsrobustheit |
| Farbanpassung | Verschiedene Belichtungen → Helligkeitsrobustheit |
| Leichte Rotation | Verschiedene Kamerawinkel → Winkelrobustheit |

Das Modell sieht nie zweimal exakt dasselbe Bild – und kann sich so nicht darauf einprägen.

⚠️ **Wichtig:** Augmentation gilt **nur für die Trainingsdaten** – Validierungsdaten bleiben unverändert!

In [ ]:
# Augmentation-Pipeline definieren
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Visualisierung: gleiches Originalbild, viele verschiedene Varianten
BILD_IDX = 3   # ← anderer Index = anderes Bild
orig_pil, orig_label = full_train_raw[BILD_IDX]

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i == 0:
        ax.imshow(orig_pil)
        ax.set_title("Original", fontsize=8, color='steelblue', fontweight='bold')
    else:
        aug = transform_aug(orig_pil)
        img = (aug.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
        ax.imshow(img)
        ax.set_title(f"Var. {i}", fontsize=7, color='gray')
    ax.axis('off')

plt.suptitle(f"Data Augmentation: 32 zufällige Varianten desselben Bildes "
             f"(Klasse: {KLASSEN_DE[orig_label]})", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gleiches Subset (500 Bilder) – jetzt MIT Augmentation
full_train_aug  = torchvision.datasets.CIFAR10('./data', train=True, download=False, transform=transform_aug)
train_loader_aug = DataLoader(Subset(full_train_aug, range(N_TRAIN)), batch_size=32, shuffle=True, num_workers=2)

print("=== Training MIT Augmentation  (dieselben 500 Bilder!) ===")
modell_2 = KleinesCNN().to(device)
v2 = trainiere(modell_2, train_loader_aug, val_loader, n_epochs=N_EPOCHS)

In [ ]:
# Direkter Vergleich: ohne vs. mit Augmentation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ep = range(1, N_EPOCHS + 1)

# --- Loss-Vergleich ---
ax1.plot(ep, v1['train_loss'], '--', color='steelblue', alpha=0.5, lw=1.5, label='Train (ohne Aug.)')
ax1.plot(ep, v1['val_loss'],         color='steelblue',             lw=2.2, label='Val   (ohne Aug.)')
ax1.plot(ep, v2['train_loss'], '--', color='tomato',    alpha=0.5, lw=1.5, label='Train (mit Aug.)')
ax1.plot(ep, v2['val_loss'],         color='tomato',                lw=2.2, label='Val   (mit Aug.)')
ax1.set_xlabel('Epoche'); ax1.set_ylabel('Loss')
ax1.set_title('Loss-Vergleich'); ax1.legend(); ax1.grid(alpha=0.3)

# --- Validierungs-Accuracy-Vergleich ---
va1 = [a * 100 for a in v1['val_acc']]
va2 = [a * 100 for a in v2['val_acc']]
ax2.plot(ep, va1, color='steelblue', lw=2.5, label='Val Acc. (ohne Aug.)')
ax2.plot(ep, va2, color='tomato',    lw=2.5, label='Val Acc. (mit Aug.)')
ax2.fill_between(ep, va1, va2, where=[a2 > a1 for a1, a2 in zip(va1, va2)],
                 alpha=0.15, color='green', label='Verbesserung durch Aug.')
ax2.set_xlabel('Epoche'); ax2.set_ylabel('Accuracy (%)'); ax2.set_ylim(0, 60)
ax2.set_title('Validierungs-Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Vergleich: Ohne vs. Mit Data Augmentation  (je 500 Trainingsbilder)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

a1_fin = v1['val_acc'][-1]
a2_fin = v2['val_acc'][-1]
print(f"\n📊 Finale Validierungs-Accuracy:")
print(f"  Ohne Augmentation: {a1_fin:.1%}")
print(f"  Mit  Augmentation: {a2_fin:.1%}")
print(f"  Verbesserung:      +{a2_fin - a1_fin:.1%} Prozentpunkte  – bei identischer Datenmenge!")

### 🔍 Interpretation

- Die **Lücke zwischen Training und Validierung** ist mit Augmentation kleiner → weniger Overfitting
- Die **Validierungs-Accuracy ist höher** – mit denselben 500 Originalbildern!
- Das Modell sieht nie zweimal exakt dasselbe Bild und kann sich nicht auf einzelne Bilder einprägen

**Data Augmentation ist eine der einfachsten und wirksamsten Massnahmen gegen Overfitting.**  
Sie ist heute Standard in jeder professionellen CV-Pipeline – unabhängig von der Datenmenge.

---
## Teil 3: SOTA-Modelle – auf den Schultern von Giganten

### Warum nicht immer selbst trainieren?

Ein gutes Bildklassifikationsmodell von Grund auf zu trainieren braucht:
- **Millionen** gelabelter Bilder
- **Tage bis Wochen** Rechenzeit auf vielen GPUs
- Jahrelange Forschungs- und Engineering-Erfahrung

Diese Arbeit wurde bereits geleistet. Auf Plattformen wie **Hugging Face** und **Ultralytics**  
stehen SOTA-Modelle frei zur Verfügung – nutzbar in wenigen Zeilen Code.

### Zwei Nutzungsstrategien

| Strategie | Beschreibung | Wann sinnvoll? |
|---|---|---|
| **Zero-Shot / Direkte Nutzung** | Vortrainiertes Modell direkt für Inferenz | Klassen überlappen mit Trainingsdaten |
| **Fine-Tuning** | Letzte Schichten auf eigene Daten anpassen | Eigene, spezifische Klassen |

Wir zeigen **direkte Nutzung** – kein weiteres Training notwendig.

### 3a) Hugging Face – Bildklassifikation mit einem Vision Transformer (ViT)

[Hugging Face](https://huggingface.co) ist die grösste Plattform für vortrainierte KI-Modelle (über 1 Million Modelle).

Wir verwenden **`google/vit-base-patch16-224`**:
- **ViT** (Vision Transformer): Teilt das Bild in 16×16 Pixel grosse Patches auf und verarbeitet sie  
  wie Wörter in einem Sprachmodell (Attention-Mechanismus)
- Vortrainiert auf **ImageNet-21k** (14 Millionen Bilder, 21 000 Klassen)
- Fine-getuned auf **ImageNet-1k** (1 000 Klassen)

Mit der `pipeline`-Abstraktion von Hugging Face reichen **drei Zeilen** für State-of-the-Art-Klassifikation.

In [ ]:
from transformers import pipeline

print("Lade ViT-Modell von Hugging Face (erster Download: ~330 MB)...")
vit = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    device=0 if torch.cuda.is_available() else -1
)
print("Modell bereit!\n")

# Je ein Testbild pro CIFAR-10-Klasse aus dem Validierungsset
klassen_idx = {}
for i in range(len(full_val_raw)):
    _, lbl = full_val_raw[i]
    if lbl not in klassen_idx:
        klassen_idx[lbl] = i
    if len(klassen_idx) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for k in range(10):
    ax   = axes[k // 5][k % 5]
    img_pil, true_lbl = full_val_raw[klassen_idx[k]]

    # CIFAR-10-Bilder sind 32x32 – ViT erwartet 224x224
    img_gross = img_pil.resize((224, 224), Image.BILINEAR)

    resultate = vit(img_gross)
    top1_label = resultate[0]['label'].split(',')[0]
    top1_score = resultate[0]['score']

    # Treffer: prüfe ob englische Klasse im Top-1 Label vorkommt
    korrekt = KLASSEN_EN[true_lbl] in resultate[0]['label'].lower()
    farbe   = '#2a9d8f' if korrekt else '#e76f51'

    ax.imshow(img_gross)
    ax.set_title(
        f"Wahr:  {KLASSEN_DE[true_lbl]}\n"
        f"ViT:   {top1_label}\n"
        f"({top1_score:.0%})",
        fontsize=8, color=farbe
    )
    ax.axis('off')

plt.suptitle(
    "ViT-Klassifikation auf CIFAR-10-Testbildern\n"
    "Grün = korrekte Klasse (Top-1)   Rot = Fehlklassifikation\n"
    "Hinweis: ViT wurde auf hochaufgelösten Bildern trainiert – 32×32 ist eine Herausforderung!",
    fontsize=10, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
def klassifiziere_url(url: str):
    """Lädt ein Bild von einer URL und klassifiziert es mit ViT."""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, timeout=15, headers=headers)
        resp.raise_for_status()
        buf = BytesIO(resp.content)
        buf.seek(0)  # Position zurücksetzen
        img = Image.open(buf).convert("RGB")
        res = vit(img.resize((224, 224), Image.BILINEAR))
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        labels = "\n".join([f"{r['label'].split(',')[0]}: {r['score']:.1%}" for r in res[:5]])
        ax.set_title(f"Top-5 Vorhersagen:\n{labels}", fontsize=9, family='monospace')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Fehler: {e}")

# ← URL nach Belieben ändern
klassifiziere_url("https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=640")  # Hund

### 3b) timm – Über 700 vortrainierte Modelle in einem Paket

`timm` (PyTorch Image Models) ist die umfassendste Sammlung vortrainierter Vision-Modelle.  
Von klassischen CNNs bis zu modernen Transformern – mit einer einheitlichen API.

In [ ]:
import timm

alle = timm.list_models(pretrained=True)
print(f"Verfügbare vortrainierte Modelle in timm: {len(alle)}\n")

familien = [
    ('resnet',      'Klassisches CNN             (He et al., 2015)'),
    ('efficientnet','Skaliertes CNN              (Tan & Le, 2019)'),
    ('vit_',        'Vision Transformer          (Dosovitskiy, 2020)'),
    ('convnext',    'Modernisiertes CNN          (Liu et al., 2022)'),
    ('swin',        'Swin Transformer            (Liu et al., 2021)'),
]

print(f"{'Familie':<22} {'Modelle':>7}   Beschreibung")
print("-" * 70)
for prefix, beschr in familien:
    n   = len([m for m in alle if m.startswith(prefix)])
    bsp = [m for m in alle if m.startswith(prefix)][0]
    print(f"  {prefix:<20} {n:>6}   {beschr}")
    print(f"  {'':20}          z.B. → {bsp}")

print()
print("📌 Nutzung in 2 Zeilen Code:")
print('   modell = timm.create_model("convnext_base", pretrained=True, num_classes=0)')
print('   features = modell(bild_tensor)   # Feature-Vektor (kein weiteres Training nötig!)')

### 3c) Ultralytics YOLO – Echtzeit-Objekterkennung

**YOLO** (You Only Look Once) ist die bekannteste Familie für Objekterkennung.  
Im Gegensatz zur reinen Bildklassifikation findet YOLO gleichzeitig:
- **Was** ist im Bild (Klasse)
- **Wo** ist es (Bounding Box mit Koordinaten)
- **Wie sicher** ist die Vorhersage (Konfidenz-Score)

Trainiert auf **COCO** (Common Objects in Context) – 80 Alltagsobjekte, ~200 000 gelabelte Bilder.

#### Modellgrössen – Speed vs. Accuracy Tradeoff

| Modell | Parameter | mAP (COCO) | Latenz |
|--------|-----------|------------|--------|
| YOLOv8n (nano)   | 3.2M  | 37.3 | ⚡⚡⚡⚡⚡ |
| YOLOv8s (small)  | 11.2M | 44.9 | ⚡⚡⚡⚡  |
| YOLOv8m (medium) | 25.9M | 50.2 | ⚡⚡⚡   |
| YOLOv8l (large)  | 43.7M | 52.9 | ⚡⚡    |
| YOLOv8x (extra)  | 68.2M | 53.9 | ⚡     |

*mAP = mean Average Precision – höher ist besser*

In [ ]:
from ultralytics import YOLO

print("Lade YOLOv8n (~6 MB)...")
yolo = YOLO('yolov8n.pt')
print(f"Modell geladen – trainiert auf {len(yolo.names)} COCO-Klassen.")
print(f"Beispiel-Klassen: {list(yolo.names.values())[:12]} ...")

In [ ]:
# Objekterkennung auf Beispielbildern
BILDER = {
    "Strassenszene": "https://ultralytics.com/images/bus.jpg",
    "Sportszene":    "https://ultralytics.com/images/zidane.jpg",
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, (titel, url) in zip(axes, BILDER.items()):
    try:
        resp    = requests.get(url, timeout=20)
        img_pil = Image.open(BytesIO(resp.content)).convert("RGB")
        res     = yolo(img_pil, verbose=False)[0]

        # Annotiertes Bild (YOLO gibt BGR-Array zurück → in RGB umwandeln)
        ax.imshow(res.plot()[:, :, ::-1])

        # Objekte zählen
        zähler = {}
        for box in res.boxes:
            k = yolo.names[int(box.cls)]
            zähler[k] = zähler.get(k, 0) + 1
        zusammenfassung = "  ".join([f"{n}×{k}" for k, n in sorted(zähler.items())])

        ax.set_title(f"{titel}\n{len(res.boxes)} Objekte: {zusammenfassung}", fontsize=10)
    except Exception as e:
        ax.text(0.5, 0.5, f"Fehler:\n{e}", ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
        ax.set_title(titel)
    ax.axis('off')

plt.suptitle("YOLOv8n – Echtzeit-Objekterkennung", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Eigenes Bild testen – URL hier anpassen!
EIGENE_URL = "https://ultralytics.com/images/bus.jpg"

try:
    resp    = requests.get(EIGENE_URL, timeout=20)
    img_pil = Image.open(BytesIO(resp.content)).convert("RGB")
    res     = yolo(img_pil, verbose=False)[0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    ax1.imshow(img_pil)
    ax1.set_title("Originalbild", fontsize=12); ax1.axis('off')
    ax2.imshow(res.plot()[:, :, ::-1])
    ax2.set_title(f"YOLO-Ergebnis: {len(res.boxes)} Objekte erkannt", fontsize=12)
    ax2.axis('off')
    plt.tight_layout(); plt.show()

    print("\n📦 Alle erkannten Objekte (nach Konfidenz sortiert):")
    print(f"  {'Klasse':<22} {'Konfidenz':>10}   Bounding Box")
    print("  " + "-" * 60)
    for box in sorted(res.boxes, key=lambda b: float(b.conf), reverse=True):
        cls  = yolo.names[int(box.cls)]
        conf = float(box.conf)
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        print(f"  {cls:<22} {conf:>9.1%}   [{x1}, {y1}, {x2}, {y2}]")
except Exception as e:
    print(f"Fehler: {e}")

---
## Zusammenfassung

| Konzept | Kernbotschaft | Praxis-Relevanz |
|---|---|---|
| **Overfitting** | Train Loss ↓, Val Loss ↑ → Modell lernt auswendig | Betrifft jedes Projekt mit wenig Daten |
| **Data Augmentation** | Gleiche Bilder, mehr Varianten → bessere Generalisierung | Standard in allen CV-Pipelines |
| **Hugging Face (ViT)** | 3 Zeilen Code für Weltklasse-Klassifikation | Direktnutzung oder als Basis für Fine-Tuning |
| **Ultralytics YOLO** | Klasse + Position in Echtzeit | Sicherheit, Robotik, Retail, Medizin, ... |

---

### Weiterführende Links

- [Hugging Face Model Hub – Image Classification](https://huggingface.co/models?pipeline_tag=image-classification)
- [timm Model Zoo Dokumentation](https://timm.fast.ai/)
- [Ultralytics YOLO Dokumentation](https://docs.ultralytics.com/)
- [Papers With Code – ImageNet Leaderboard](https://paperswithcode.com/sota/image-classification-on-imagenet)
- [Torchvision Transforms](https://pytorch.org/vision/stable/transforms.html)